In [5]:
import pandas as pd
import numpy as np

# Load the file - we will guess header row is row 3 (0-indexed: 2)
# Let's first read without skipping rows to see the structure
file_path = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\raw\ncrb\2022\district_crime_women_2022.xlsx"

# Read first 10 rows to inspect structure
df_raw = pd.read_excel(file_path, header=None, nrows=15)

print("Shape:", df_raw.shape)
print("\nFirst 15 rows:")
df_raw

Shape: (15, 54)

First 15 rows:


,0,1,2,3,4,5,6,7,8,9,...,44,45,46,47,48,49,50,51,52,53
0,Districtwise Crime against Women - 2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S. No,State/District,Murder with Rape/Gang Rape,Dowry Deaths (Sec. 304B IPC),Abetment to Suicide of Women (Sec. 305/306 IPC),Miscarriage (Sec. 313 & 314 IPC),Acid Attack (Sec. 326A IPC),Attempt to Acid Attack (Sec. 326B IPC),Cruelty by Husband or his relatives (Sec. 498 ...,Kidnapping & Abduction of Women,...,NaN,Protection of Children from Sexual Violence Ac...,NaN,NaN,NaN,NaN,NaN,NaN,Indecent Representation of Women (Prohibition)...,Total Crime against Women (IPC+SLL)
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kidnapping & Abduction of Women (Total) (Col.1...,...,Other Women Centric Cyber Crimes (Ex. Blackmai...,Protection of Children from Sexual Violence Ac...,Child Rape (Sec. 4 & 6 of POCSO Act) / Sec. 37...,Sexual Assault of Children (Sec. 8 & 10 of POC...,Sexual Harassment (Sec. 12 of POCSO Act) / Sec...,Use of Child for Pornography/Storing Child Por...,POCSO Act (Sections 17 to 22) / Other offences...,POCSO Act r/w Section 377 IPC / Unnatural Off...,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,2,3,4,5,6,7,8,9,10,...,45,46,47,48,49,50,51,52,53,54
5,State: Andhra Pradesh,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,Alluri Sitharama Raju,1,2,2,0,0,0,58,0,...,0,13,13,0,0,0,0,0,0,138
7,2,Anakapalli,0,5,4,0,0,0,276,25,...,0,58,32,26,0,0,0,0,0,709
8,3,Anantapuramu,0,2,19,0,0,0,343,21,...,1,65,34,4,27,0,0,0,0,709
9,4,Annamayya,0,2,6,0,0,0,362,2,...,0,37,27,5,5,0,0,0,0,611


In [14]:
# 1. Load the raw data without skipping ANY rows
df_raw = pd.read_excel(file_path)

# 2. CREATE A CLEAN STATE COLUMN BY SEARCHING EVERY CELL
def extract_state_anywhere(row):
    for cell in row:
        s_cell = str(cell)
        if 'State:' in s_cell:
            return s_cell.replace('State:', '').strip()
    return np.nan

# Apply the search across the whole row
df_raw['State_Final'] = df_raw.apply(extract_state_anywhere, axis=1)

# 3. Forward Fill the State names
df_raw['State_Final'] = df_raw['State_Final'].ffill()

# 4. Now we need to find where the actual data starts
# We look for the row that has "S. No" or "S.No"
# Then we take everything below that
data_start_idx = 0
for i, row in df_raw.iterrows():
    if 'S. No' in str(row.values) or 'S.No' in str(row.values):
        data_start_idx = i
        break

# 5. Slice the data from the start point
df_subset = df_raw.iloc[data_start_idx + 1:].copy()

# 6. Select the columns based on the structure we know
# We'll use the column indices since the names are messy
# 0: S.No, 1: District, 2: Murder, 3: Dowry, 4: Suicide, 8: Cruelty, 9: Kidnap, 23: Rape, -1: State_Final
df_clean = df_subset.iloc[:, [0, 1, 2, 3, 4, 8, 9, 23, -1]].copy()
df_clean.columns = ['S_No', 'District', 'Murder_Rape', 'Dowry_Death', 'Abetment_Suicide', 'Cruelty_By_Husband', 'Kidnap_Total', 'Rape_Total', 'State']

# 7. Filter: Keep only rows where S_No is a number
# We use a try-except to handle cases where S_No might be a string
def is_it_number(x):
    try:
        float(x)
        return True
    except:
        return False

df_final = df_clean[df_clean['S_No'].apply(is_it_number)].copy()

# 8. Drop S_No and Reorder
df_final = df_final.dropna(subset=['S_No']) # Remove any remaining NaNs in S_No
df_final = df_final[['State', 'District', 'Murder_Rape', 'Dowry_Death', 'Abetment_Suicide', 'Cruelty_By_Husband', 'Kidnap_Total', 'Rape_Total']]

# 9. Final numeric conversion
crime_cols = ['Murder_Rape', 'Dowry_Death', 'Abetment_Suicide', 'Cruelty_By_Husband', 'Kidnap_Total', 'Rape_Total']
for col in crime_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)

print("Nuclear Cleaning Completed!")
print("Cleaned Shape:", df_final.shape)
display(df_final.head(20))

Nuclear Cleaning Completed!
Cleaned Shape: (935, 8)


,State,District,Murder_Rape,Dowry_Death,Abetment_Suicide,Cruelty_By_Husband,Kidnap_Total,Rape_Total
3,NaN,2,3,4,5,9,10,24
5,Andhra Pradesh,Alluri Sitharama Raju,1,2,2,58,0,34
6,Andhra Pradesh,Anakapalli,0,5,4,276,25,37
7,Andhra Pradesh,Anantapuramu,0,2,19,343,21,12
8,Andhra Pradesh,Annamayya,0,2,6,362,2,11
9,Andhra Pradesh,Bapatla,0,3,9,447,6,22
10,Andhra Pradesh,Chittoor,0,2,17,357,7,12
11,Andhra Pradesh,Dr BR Ambedkar Konaseema,0,3,4,386,22,23
12,Andhra Pradesh,East Godavari,1,0,12,672,22,27
13,Andhra Pradesh,Eluru,0,8,16,653,45,28


In [15]:
# Remove rows where the District column contains only numbers (these are the leftover header rows)
df_final = df_final[~df_final['District'].astype(str).str.isnumeric()]

# Reset index for a clean look
df_final = df_final.reset_index(drop=True)

print("Cleaned successfully! Junk row removed.")
display(df_final.head(10))

Cleaned successfully! Junk row removed.


,State,District,Murder_Rape,Dowry_Death,Abetment_Suicide,Cruelty_By_Husband,Kidnap_Total,Rape_Total
0,Andhra Pradesh,Alluri Sitharama Raju,1,2,2,58,0,34
1,Andhra Pradesh,Anakapalli,0,5,4,276,25,37
2,Andhra Pradesh,Anantapuramu,0,2,19,343,21,12
3,Andhra Pradesh,Annamayya,0,2,6,362,2,11
4,Andhra Pradesh,Bapatla,0,3,9,447,6,22
5,Andhra Pradesh,Chittoor,0,2,17,357,7,12
6,Andhra Pradesh,Dr BR Ambedkar Konaseema,0,3,4,386,22,23
7,Andhra Pradesh,East Godavari,1,0,12,672,22,27
8,Andhra Pradesh,Eluru,0,8,16,653,45,28
9,Andhra Pradesh,Guntakal Railway,0,0,2,1,0,0


In [16]:
import glob

# 1. Define the Master Cleaning Function
def clean_ncrb_district_data(file_path, year):
    # Load raw
    df_raw = pd.read_excel(file_path)
    
    # Extract State names (Our Nuclear Search)
    def extract_state(row):
        for cell in row:
            if 'State:' in str(cell):
                return str(cell).replace('State:', '').strip()
        return np.nan
    
    df_raw['State'] = df_raw.apply(extract_state, axis=1)
    df_raw['State'] = df_raw['State'].ffill()
    
    # Find data start (where S. No lives)
    data_start_idx = 0
    for i, row in df_raw.iterrows():
        if 'S. No' in str(row.values) or 'S.No' in str(row.values):
            data_start_idx = i
            break
    
    # Slice and pick columns
    df_subset = df_raw.iloc[data_start_idx + 1:].copy()
    
    # Using relative indices to be safe
    # 0: S.No, 1: District, 2: Murder, 3: Dowry, 4: Suicide, 8: Cruelty, 9: Kidnap, 23: Rape, -1: State
    df_clean = df_subset.iloc[:, [0, 1, 2, 3, 4, 8, 9, 23, -1]].copy()
    df_clean.columns = ['S_No', 'District', 'Murder_Rape', 'Dowry_Death', 'Abetment_Suicide', 'Cruelty_By_Husband', 'Kidnap_Total', 'Rape_Total', 'State']
    
    # Filter for districts (S_No must be a number)
    def is_num(x):
        try: float(x); return True
        except: return False
        
    df_final = df_clean[df_clean['S_No'].apply(is_num)].copy()
    
    # Remove junk text rows from District column
    df_final = df_final[~df_final['District'].astype(str).str.isnumeric()]
    
    # Drop S_No and add the YEAR
    df_final = df_final.drop(columns=['S_No'])
    df_final['Year'] = year
    
    # Final numeric conversion
    crime_cols = ['Murder_Rape', 'Dowry_Death', 'Abetment_Suicide', 'Cruelty_By_Husband', 'Kidnap_Total', 'Rape_Total']
    for col in crime_cols:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)
        
    return df_final

# 2. CREATE A LOOP TO PROCESS ALL YEARS
all_years_data = []
years = [2021, 2022, 2023, 2024]

for yr in years:
    # Build path (Adjust this path to match your actual folder structure)
    path = rf"C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\raw\ncrb\{yr}\district_crime_women_{yr}.xlsx"
    
    try:
        print(f"Processing Year: {yr}...")
        df_temp = clean_ncrb_district_data(path, yr)
        all_years_data.append(df_temp)
        print(f"Successfully cleaned {yr} - Shape: {df_temp.shape}")
    except Exception as e:
        print(f"Error in {yr}: {e}")

# 3. COMBINE EVERYTHING
df_master = pd.concat(all_years_data, ignore_index=True)

print("\n--- ALL YEARS COMBINED ---")
print("Total Rows:", df_master.shape[0])
display(df_master.sample(10)) # Look at 10 random rows to check different years

Processing Year: 2021...
Successfully cleaned 2021 - Shape: (990, 9)
Processing Year: 2022...
Successfully cleaned 2022 - Shape: (1016, 9)
Processing Year: 2023...
Successfully cleaned 2023 - Shape: (1041, 9)
Processing Year: 2024...
Successfully cleaned 2024 - Shape: (1047, 9)

--- ALL YEARS COMBINED ---
Total Rows: 4094


,District,Murder_Rape,Dowry_Death,Abetment_Suicide,Cruelty_By_Husband,Kidnap_Total,Rape_Total,State,Year
3880,Barabanki,11.0,11.0,0.0,0.0,0.0,0.0,Uttar Pradesh,2024
3788,Pudukottai,1.0,1.0,0.0,1.0,0.0,0.0,Tamil Nadu,2024
3188,Saharsa,7.0,7.0,0.0,0.0,0.0,1.0,Bihar,2024
670,Jhalawar,0.0,6.0,9.0,434.0,193.0,123.0,Rajasthan,2021
1172,Khairagarh - Chhuikhadan-Gandai,0.0,3.0,1.0,7.0,44.0,9.0,Chhattisgarh,2022
1509,Garo Hills South,0.0,0.0,0.0,1.0,5.0,2.0,Meghalaya,2022
2493,Solapur Rural,0.0,3.0,23.0,159.0,203.0,91.0,Maharashtra,2023
813,Chandoli,0.0,6.0,1.0,115.0,37.0,17.0,Uttar Pradesh,2021
2260,Jind,0.0,12.0,9.0,241.0,90.0,43.0,Haryana,2023
498,Solapur Rural,1.0,2.0,45.0,188.0,173.0,81.0,Maharashtra,2021


In [18]:
# 1. Load the Census File
census_path = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\raw\census\district_population_2011.xlsx"
df_census_raw = pd.read_excel(census_path)

# 2. Filter for District level and 'Total' area
# Level is in Column G ('Level'), TRU is in Column I ('TRU')
df_pop = df_census_raw[(df_census_raw['Level'] == 'DISTRICT') & (df_census_raw['TRU'] == 'Total')].copy()

# 3. Select only the columns we need
# Name is Column H, Total Population is Column J
df_pop = df_pop[['Name', 'TOT_P', 'TOT_F']]

# 4. Rename columns for clarity
df_pop.columns = ['District', 'Population_2011', 'Female_Pop_2011']

# 5. Standardize District Names
# Remove extra spaces and make Uppercase
df_pop['District'] = df_pop['District'].astype(str).str.strip().str.upper()

# 6. Remove the word "DISTRICT" if it appears in the name (common in Census)
df_pop['District'] = df_pop['District'].str.replace('DISTRICT ', '').str.replace(' DISTRICT', '')

print("Census Data Cleaned!")
print("Number of Districts in Census:", len(df_pop))
display(df_pop.head(10))

Census Data Cleaned!
Number of Districts in Census: 640


,District,Population_2011,Female_Pop_2011
6,KUPWARA,870354,396164
9,BADGAM,753745,355704
12,LEH(LADAKH),133487,54516
15,KARGIL,140802,63017
18,PUNCH,476835,224936
21,RAJOURI,642415,297064
24,KATHUA,616435,290326
27,BARAMULA,1008039,473306
30,BANDIPORE,392232,184552
33,SRINAGAR,1236829,585705


In [20]:
# 1. Standardize NCRB District Names
df_master['District_Clean'] = df_master['District'].astype(str).str.strip().str.upper()

# 2. Project 2011 Population to 2024
# Average India growth is approx 1.1% per year. (1.011 ^ 13 years ≈ 1.15)
growth_factor = 1.15 
df_pop['Pop_2024_Est'] = (df_pop['Population_2011'] * growth_factor).astype(int)

# 3. Attempt the Merge
# We use a 'left' join to keep all crime records even if population is missing
df_combined = pd.merge(df_master, 
                       df_pop[['District', 'Pop_2024_Est']], 
                       left_on='District_Clean', 
                       right_on='District', 
                       how='left', 
                       suffixes=('', '_census'))

# 4. Check for Mismatches
mismatched = df_combined[df_combined['Pop_2024_Est'].isnull()]['District_Clean'].unique()

print(f"Total rows in Master: {len(df_combined)}")
print(f"Districts with NO population match: {len(mismatched)}")
print("\nFirst 10 mismatched districts (need manual mapping):")
print(mismatched[:10])

# 5. Drop the extra column from the join
df_combined = df_combined.drop(columns=['District_census'])

display(df_combined.head(20))

Total rows in Master: 4149
Districts with NO population match: 543

First 10 mismatched districts (need manual mapping):
<StringArray>
[                 nan,           'CUDDAPAH',   'GUNTAKAL RAILWAY',
       'GUNTUR URBAN',            'NELLORE',          'PRAKASHAM',
        'RAJAHMUNDRY',    'TIRUPATHI URBAN',    'VIJAYAWADA CITY',
 'VIJAYAWADA RAILWAY']
Length: 10, dtype: str


,District,Murder_Rape,Dowry_Death,Abetment_Suicide,Cruelty_By_Husband,Kidnap_Total,Rape_Total,State,Year,District_Clean,Pop_2024_Est
0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2021,NaN,NaN
1,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2021,NaN,NaN
2,Anantapur,0.0,9.0,37.0,375.0,50.0,21.0,Andhra Pradesh,2021,ANANTAPUR,4693320.0
3,Chittoor,0.0,18.0,25.0,321.0,18.0,13.0,Andhra Pradesh,2021,CHITTOOR,4800173.0
4,Cuddapah,0.0,3.0,5.0,330.0,18.0,8.0,Andhra Pradesh,2021,CUDDAPAH,NaN
5,East Godavari,0.0,11.0,26.0,442.0,46.0,70.0,Andhra Pradesh,2021,EAST GODAVARI,5927440.0
6,Guntakal Railway,0.0,1.0,1.0,2.0,0.0,0.0,Andhra Pradesh,2021,GUNTAKAL RAILWAY,NaN
7,Guntur,1.0,3.0,44.0,614.0,67.0,16.0,Andhra Pradesh,2021,GUNTUR,5620984.0
8,Guntur Urban,0.0,7.0,16.0,257.0,39.0,26.0,Andhra Pradesh,2021,GUNTUR URBAN,NaN
9,Krishna,0.0,7.0,24.0,372.0,62.0,27.0,Andhra Pradesh,2021,KRISHNA,5195007.0


In [21]:
# =================================================================
# HEADING: ADVANCED DISTRICT NAME STANDARDIZATION & POPULATION JOIN
# =================================================================

import re

def super_clean_name(text):
    """Function to strip everything except alphabets to improve matching"""
    text = str(text).upper()
    text = re.sub(r'\(.*\)', '', text) # Remove brackets and content inside
    text = re.sub(r'[^A-Z\s]', '', text) # Remove special chars
    text = text.replace('DISTRICT', '').replace('CITY', '').replace('RURAL', '').replace('URBAN', '')
    return " ".join(text.split()) # Remove extra whitespaces

# 1. Standardize names in both dataframes
df_master['Dist_Match'] = df_master['District_Clean'].apply(super_clean_name)
df_pop['Dist_Match'] = df_pop['District'].apply(super_clean_name)

# 2. Define a manual mapping for common large mismatches
mapping_dict = {
    'GURGAON': 'GURUGRAM',
    'BANGALORE': 'BENGALURU',
    'BOMBAY': 'MUMBAI',
    'MADRAS': 'CHENNAI',
    'ALLAHABAD': 'PRAYAGRAJ'
}
df_master['Dist_Match'] = df_master['Dist_Match'].replace(mapping_dict)

# 3. Perform the merge again
df_combined = pd.merge(df_master, 
                       df_pop[['Dist_Match', 'Pop_2024_Est']], 
                       on='Dist_Match', 
                       how='left')

# 4. Fill remaining NaNs with the STATE AVERAGE population
# This ensures new districts get a 'fair' population estimate
state_avg_pop = df_combined.groupby('State')['Pop_2024_Est'].transform('mean')
df_combined['Pop_2024_Est'] = df_combined['Pop_2024_Est'].fillna(state_avg_pop)

# 5. Final fallback: If some states still have NaNs, use National Average
national_avg = df_combined['Pop_2024_Est'].mean()
df_combined['Pop_2024_Est'] = df_combined['Pop_2024_Est'].fillna(national_avg).astype(int)

# =================================================================
# HEADING: CRIME RATE CALCULATION (Per 100,000 People)
# =================================================================

# We sum the key crimes to get a 'Total_Violence' score
df_combined['Total_Violence'] = (df_combined['Murder_Rape'] + 
                                df_combined['Dowry_Death'] + 
                                df_combined['Cruelty_By_Husband'] + 
                                df_combined['Rape_Total'])

# Formula: (Crimes / Population) * 100,000
df_combined['Crime_Rate'] = (df_combined['Total_Violence'] / df_combined['Pop_2024_Est']) * 100000

print("Cleaning and Normalization Complete!")
print(f"Any remaining NaNs in Population: {df_combined['Pop_2024_Est'].isnull().sum()}")
display(df_combined[['State', 'District', 'Pop_2024_Est', 'Total_Violence', 'Crime_Rate']].head(10))

Cleaning and Normalization Complete!
Any remaining NaNs in Population: 0


,State,District,Pop_2024_Est,Total_Violence,Crime_Rate
0,NaN,NaN,2086330,0.0,0.000000
1,NaN,NaN,2086330,0.0,0.000000
2,Andhra Pradesh,Anantapur,4693320,405.0,8.629286
3,Andhra Pradesh,Chittoor,4800173,352.0,7.333069
4,Andhra Pradesh,Cuddapah,4427452,341.0,7.701947
5,Andhra Pradesh,East Godavari,5927440,523.0,8.823371
6,Andhra Pradesh,Guntakal Railway,4427452,3.0,0.067759
7,Andhra Pradesh,Guntur,5620984,634.0,11.279164
8,Andhra Pradesh,Guntur Urban,5620984,290.0,5.159239
9,Andhra Pradesh,Krishna,5195007,406.0,7.815196


In [36]:
# =================================================================
# HEADING: REMOVING JUNK ROWS AND PREPARING FOR FINAL JOIN
# =================================================================

# 1. Remove rows where State or District is NaN (Cleaning the first 2 rows you saw)
df_combined = df_combined.dropna(subset=['State', 'District']).copy()

# 2. Standardize State names for joining
df_combined['State_Match'] = df_combined['State'].str.strip().str.upper()

# =================================================================
# HEADING: INTEGRATING POLICE STRENGTH DATA
# =================================================================

# Load Police Data
police_path = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\raw\police\police_strength_statewise.csv"
df_police = pd.read_csv(police_path)
df_police['State_Match'] = df_police['State'].str.strip().str.upper()

# Join Police data to our master
df_final = pd.merge(df_combined, 
                    df_police[['State_Match', 'Sanctioned_Strength', 'Actual_Strength']], 
                    on='State_Match', 
                    how='left')

# =================================================================
# HEADING: INTEGRATING NFHS-5 UNDERREPORTING DATA
# =================================================================

# Load the new NFHS CSV you created
nfhs_path = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\raw\nfhs\nfhs5_state_data.csv"
df_nfhs = pd.read_csv(nfhs_path)
df_nfhs['State_Match'] = df_nfhs['State'].str.strip().str.upper()

# Join NFHS data
df_final = pd.merge(df_final, 
                    df_nfhs[['State_Match', 'Spousal_Violence_Percent', 'Sought_Help_Percent']], 
                    on='State_Match', 
                    how='left')

# =================================================================
# HEADING: FINAL METRIC CALCULATIONS & EXPORT
# =================================================================

# 1. Police Vacancy Rate
df_final['Police_Vacancy_Rate'] = (df_final['Sanctioned_Strength'] - df_final['Actual_Strength']) / df_final['Sanctioned_Strength']

# 2. Fill any missing values in metrics with 0 or averages
df_final['Spousal_Violence_Percent'] = df_final['Spousal_Violence_Percent'].fillna(df_final['Spousal_Violence_Percent'].mean())
df_final['Police_Vacancy_Rate'] = df_final['Police_Vacancy_Rate'].fillna(0)

# 3. Select final column structure
final_cols = ['Year', 'State', 'District', 'Pop_2024_Est', 
              'Total_Violence', 'Crime_Rate', 
              'Police_Vacancy_Rate', 'Spousal_Violence_Percent', 'Sought_Help_Percent']

df_master_clean = df_final[final_cols].copy()

# 4. Save the Final Master File
output_csv = r"C:\Users\diyas\OneDrive\Desktop\women_safety_india\data\cleaned\master_crime_data_clean.csv"
df_master_clean.to_csv(output_csv, index=False)

print("--- STEP 2: DATA CLEANING & INTEGRATION SUCCESSFUL ---")
print(f"Final Data Shape: {df_master_clean.shape}")
display(df_master_clean.head(10))

--- STEP 2: DATA CLEANING & INTEGRATION SUCCESSFUL ---
Final Data Shape: (3976, 9)


,Year,State,District,Pop_2024_Est,Total_Violence,Crime_Rate,Police_Vacancy_Rate,Spousal_Violence_Percent,Sought_Help_Percent
0,2021,Andhra Pradesh,Anantapur,4693320,405.0,8.629286,0.171097,30.0,4.4
1,2021,Andhra Pradesh,Chittoor,4800173,352.0,7.333069,0.171097,30.0,4.4
2,2021,Andhra Pradesh,Cuddapah,4427452,341.0,7.701947,0.171097,30.0,4.4
3,2021,Andhra Pradesh,East Godavari,5927440,523.0,8.823371,0.171097,30.0,4.4
4,2021,Andhra Pradesh,Guntakal Railway,4427452,3.0,0.067759,0.171097,30.0,4.4
5,2021,Andhra Pradesh,Guntur,5620984,634.0,11.279164,0.171097,30.0,4.4
6,2021,Andhra Pradesh,Guntur Urban,5620984,290.0,5.159239,0.171097,30.0,4.4
7,2021,Andhra Pradesh,Krishna,5195007,406.0,7.815196,0.171097,30.0,4.4
8,2021,Andhra Pradesh,Kurnool,4661482,364.0,7.808675,0.171097,30.0,4.4
9,2021,Andhra Pradesh,Nellore,4427452,387.0,8.740919,0.171097,30.0,4.4
